In [45]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from pathlib import Path
from pprint import pprint
from collections import Counter
from typing import Dict, List
import json

from presidio_evaluator import InputSample
from presidio_evaluator.evaluation import SpanEvaluator, ModelError, Plotter
from presidio_evaluator.experiment_tracking import get_experiment_tracker

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

%reload_ext autoreload
%autoreload 2


dataset_name = "data_person_1000_zh_fixed_gender.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd(), "data", dataset_name), token_model_version="zh_core_web_sm")
print(len(dataset))

def get_entity_counts(dataset: List[InputSample]) -> Dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter


entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print("\nMin and max number of tokens in dataset: "\
f"Min: {min([len(sample.tokens) for sample in dataset])}, "\
f"Max: {max([len(sample.tokens) for sample in dataset])}")

print(f"Min and max sentence length in dataset: " \
f"Min: {min([len(sample.full_text) for sample in dataset])}, "\
f"Max: {max([len(sample.full_text) for sample in dataset])}")

tokenizing input: 100%|██████████| 975/975 [00:16<00:00, 58.35it/s]


975
Count per entity:
[('O', 78240), ('STREET_ADDRESS', 5497), ('PERSON', 3394),
 ('EMAIL_ADDRESS', 3385), ('AGE', 1478), ('ID', 975), ('PHONE_NUMBER', 974),
 ('GENDER', 928), ('CREDIT_CARD', 53)]

Min and max number of tokens in dataset: Min: 77, Max: 126
Min and max sentence length in dataset: Min: 180, Max: 259


In [46]:
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
from presidio_analyzer.nlp_engine import NlpEngineProvider
from hanlp_engine import HanLPNlpEngine, HanLPRecognizer
from zh_pattern_recognizers import register_zh_pattern_recognizers
import hanlp

tok = hanlp.load(hanlp.pretrained.tok.COARSE_ELECTRA_SMALL_ZH)
ner = hanlp.load(hanlp.pretrained.ner.MSRA_NER_ELECTRA_SMALL_ZH)
hanlp_model = hanlp.pipeline().append(tok, output_key="tok").append(
    ner, input_key="tok", output_key="ner"
)

nlp_engine = HanLPNlpEngine(hanlp_model)
recognizer = HanLPRecognizer(nlp_engine)

registry = RecognizerRegistry(supported_languages=["zh"])
registry.add_recognizer(recognizer)

register_zh_pattern_recognizers(registry)

analyzer_engine = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=["zh"], registry=registry)

pprint(f"Supported entities for Chinese:")
pprint(analyzer_engine.get_supported_entities("zh"), compact=True)

print(f"\nLoaded recognizers for Chinese:")
pprint([rec.name for rec in analyzer_engine.registry.get_recognizers("zh", all_fields=True)], compact=True)

print(f"\nLoaded NER models:")
# pprint(analyzer_engine.nlp_engine.models)


'Supported entities for Chinese:'
['PHONE_NUMBER', 'EMAIL_ADDRESS', 'LOCATION', 'AGE', 'GENDER', 'PERSON', 'ID']

Loaded recognizers for Chinese:
['HanLPRecognizer', 'EmailRecognizer', 'PhoneRecognizer', 'IdRecognizer',
 'GenderRecognizer']

Loaded NER models:


In [47]:
from presidio_evaluator.models import PresidioAnalyzerWrapper

entities_mapping=PresidioAnalyzerWrapper.presidio_entities_map # default mapping
# Add titles and zip codes as we have recognizers for those
entities_mapping["GENDER"] = "GENDER"
# entities_mapping["STREET_ADDRESS"] = "LOCATION"
print("Entities mapping:")
pprint(entities_mapping)

dataset = SpanEvaluator.align_entity_types(
    dataset, 
    entities_mapping=entities_mapping, 
    allow_missing_mappings=True
)
new_entity_counts = get_entity_counts(dataset)
print("\nCount per entity after alignment:")
pprint(new_entity_counts.most_common(), compact=True)

dataset_entities = list(new_entity_counts.keys())

Entities mapping:
{'ADDRESS': 'LOCATION',
 'AGE': 'AGE',
 'BIRTHDAY': 'DATE_TIME',
 'CITY': 'LOCATION',
 'CREDIT_CARD': 'CREDIT_CARD',
 'CREDIT_CARD_NUMBER': 'CREDIT_CARD',
 'DATE': 'DATE_TIME',
 'DATE_OF_BIRTH': 'DATE_TIME',
 'DATE_TIME': 'DATE_TIME',
 'DOB': 'DATE_TIME',
 'DOMAIN': 'URL',
 'DOMAIN_NAME': 'URL',
 'EMAIL': 'EMAIL_ADDRESS',
 'EMAIL_ADDRESS': 'EMAIL_ADDRESS',
 'FACILITY': 'LOCATION',
 'FIRST_NAME': 'PERSON',
 'GENDER': 'GENDER',
 'GPE': 'LOCATION',
 'HCW': 'PERSON',
 'HOSP': 'ORGANIZATION',
 'HOSPITAL': 'ORGANIZATION',
 'IBAN': 'IBAN_CODE',
 'IBAN_CODE': 'IBAN_CODE',
 'ID': 'ID',
 'IP_ADDRESS': 'IP_ADDRESS',
 'LAST_NAME': 'PERSON',
 'LOC': 'LOCATION',
 'LOCATION': 'LOCATION',
 'NAME': 'PERSON',
 'NATIONALITY': 'NRP',
 'NORP': 'NRP',
 'NRP': 'NRP',
 'O': 'O',
 'ORG': 'ORGANIZATION',
 'ORGANIZATION': 'ORGANIZATION',
 'PATIENT': 'PERSON',
 'PATORG': 'ORGANIZATION',
 'PER': 'PERSON',
 'PERSON': 'PERSON',
 'PHONE': 'PHONE_NUMBER',
 'PHONE_NUMBER': 'PHONE_NUMBER',
 'PREFIX': '

In [48]:
text = "贾明川，38岁男性建筑设计师，现居住于黑龙江省哈尔滨市道里区中央大街123号。其身份证号码为230102198511153216，可通过邮箱jiamingchuan@163.com或手机13945678231联系。贾先生近期出现体重超标、呼吸短促和关节不适等症状，经哈尔滨医科大学附属第一医院诊断确诊为肥胖症。主治医师金志明为其开具了扑热息痛作为治疗方案。贾先生目前信用评分为549分，年收入100245.67元人民币，主要使用中国工商银行账户（卡号：6222021005678912345）进行资金往来。"
results = analyzer_engine.analyze(text=text, language="zh", return_decision_process=True)
for result in results:
    print(f"\nEntity: {result.entity_type}, Text: {text[result.start:result.end]}\n\nAnalysis explanation:")
    pprint(result.analysis_explanation)

print(dataset[0].full_text, results)


Entity: ID, Text: 230102198511153216

Analysis explanation:
{'recognizer': 'IdRecognizer', 'pattern_name': 'cn_id', 'pattern': '(?<!\\d)\\d{17}[\\dXx](?!\\d)', 'original_score': 0.95, 'score': 0.95, 'textual_explanation': 'Detected by `IdRecognizer` using pattern `cn_id`', 'score_context_improvement': 0, 'supportive_context_word': '', 'validation_result': None, 'regex_flags': regex.I|M|S}

Entity: PHONE_NUMBER, Text: 13945678231

Analysis explanation:
{'recognizer': 'PhoneRecognizer', 'pattern_name': 'cn_phone', 'pattern': '(?<!\\d)(?:\\+?86[- ]?)?1[3-9]\\d{9}(?!\\d)', 'original_score': 0.9, 'score': 0.9, 'textual_explanation': 'Detected by `PhoneRecognizer` using pattern `cn_phone`', 'score_context_improvement': 0, 'supportive_context_word': '', 'validation_result': None, 'regex_flags': regex.I|M|S}

Entity: PERSON, Text: 贾明川

Analysis explanation:
None

Entity: AGE, Text: 38岁

Analysis explanation:
None

Entity: LOCATION, Text: 黑龙江省

Analysis explanation:
None

Entity: LOCATION, Tex

In [49]:
# Set up the experiment tracker to log the experiment for reproducibility
experiment = get_experiment_tracker()


from hanlp_model_wrapper import HanLPModelWrapper

model = HanLPModelWrapper(
    analyzer_engine=analyzer_engine,
    labeling_scheme="IO",
    language="zh",
)

# Create the evaluator object
evaluator = SpanEvaluator(model=model, iou_threshold=0.7)


# Track model and dataset params
params = {"dataset_name": dataset_name, "model_name": evaluator.model.name}
params.update(evaluator.model.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)
experiment.log_parameter("entity_mappings", json.dumps(entities_mapping))

## Run experiment

evaluation_results = evaluator.evaluate_all(dataset, language="zh")
results = evaluator.calculate_score(evaluation_results)

# Track experiment results
experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, 
                                labels=entities)

# end experiment
experiment.end()

# Note that the experiment params and metrics are saved locally

Running model HanLPModelWrapper on dataset...


/Users/jyu36/code/ic-llm/presidio/.venv/lib/python3.12/site-packages/presidio_evaluator/evaluation/base_evaluator.py:83: UserWarning: skip words not provided, using default skip words. If you want the evaluation to not use skip words, pass skip_words=[]
  warnings.warn("skip words not provided, using default skip words. "


Finished running model on dataset
saving experiment data to /Users/jyu36/code/ic-llm/presidio/experiment_20260320-204507.json


In [50]:
from collections import Counter

predicted_label_counts = Counter(
    tag
    for r in evaluation_results
    for tag in r.predicted_tags
    if tag != "O"
)

print(predicted_label_counts)


Counter({'LOCATION': 5713, 'PERSON': 3613, 'EMAIL_ADDRESS': 3276, 'AGE': 1925, 'PHONE_NUMBER': 1264, 'GENDER': 975, 'ID': 975})


In [51]:
# # Plot output
plotter = Plotter(results=results, 
                  model_name = evaluator.model.name, 
                  save_as="svg",
                  beta = 2) 

# Path of the directory to save the plots
output_folder = Path(Path.cwd(), "plotter_output")
plotter.plot_scores(output_folder=output_folder)

In [52]:
ModelError.most_common_fp_tokens(results.model_errors)

Most common false positive tokens:
[('邮箱', 57),
 ('中国 工商 银行', 54),
 ('女士', 22),
 ('先生', 21),
 ('44 岁', 15),
 ('56 岁', 13),
 ('49 岁', 13),
 ('25 岁', 11),
 ('31 岁', 11),
 ('27 岁', 11)]
---------------
Example sentence with each FP token:
	- 邮箱 (`邮箱` pred as PHONE_NUMBER)
	- 中国 工商 银行 (`中国 工商 银行` pred as LOCATION)
	- 女士 (`女士` pred as GENDER)
	- 先生 (`先生` pred as GENDER)
	- 44 岁 (`44 岁` pred as O)
	- 56 岁 (`56 岁` pred as O)
	- 49 岁 (`49 岁` pred as O)
	- 25 岁 (`25 岁` pred as O)
	- 31 岁 (`31 岁` pred as O)
	- 27 岁 (`27 岁` pred as O)


[('邮箱', 57),
 ('中国 工商 银行', 54),
 ('女士', 22),
 ('先生', 21),
 ('44 岁', 15),
 ('56 岁', 13),
 ('49 岁', 13),
 ('25 岁', 11),
 ('31 岁', 11),
 ('27 岁', 11)]

In [53]:
fps_df = ModelError.get_fps_dataframe(results.model_errors, entity=["PHONE_NUMBER"])
fps_df[["full_text", "token", "annotation", "prediction", "sample_id"]].head(20)

,full_text,token,annotation,prediction,sample_id
0,邮箱,邮箱,O,PHONE_NUMBER,None
1,STL31071,stl31071,O,PHONE_NUMBER,None
2,邮箱ailijia,邮箱ailijia,O,PHONE_NUMBER,None
3,"1 , 023",1 023,O,PHONE_NUMBER,None
4,邮箱,邮箱,O,PHONE_NUMBER,None
5,STL26091,stl26091,O,PHONE_NUMBER,None
6,STL30091,stl30091,O,PHONE_NUMBER,None
7,STL05101,stl05101,O,PHONE_NUMBER,None
8,STL16111,stl16111,O,PHONE_NUMBER,None
9,STL29111,stl29111,O,PHONE_NUMBER,None


In [54]:
ModelError.most_common_fn_tokens(results.model_errors, n=15)

Most common false negative tokens:
[('44', 15),
 ('56', 13),
 ('49', 13),
 ('6222021234567890123', 12),
 ('25', 11),
 ('31', 11),
 ('27', 11),
 ('19', 11),
 ('54', 11),
 ('51', 10),
 ('26', 10),
 ('13857123456', 10),
 ('28', 10),
 ('76', 9),
 ('58', 9)]
---------------
Example sentence with each FN token:
	- 44 (`44` annotated as O)
	- 56 (`56` annotated as O)
	- 49 (`49` annotated as O)
	- 6222021234567890123 (`6222021234567890123` annotated as O)
	- 25 (`25` annotated as O)
	- 31 (`31` annotated as O)
	- 27 (`27` annotated as O)
	- 19 (`19` annotated as O)
	- 54 (`54` annotated as O)
	- 51 (`51` annotated as O)
	- 26 (`26` annotated as O)
	- 13857123456 (`13857123456` annotated as O)
	- 28 (`28` annotated as O)
	- 76 (`76` annotated as O)
	- 58 (`58` annotated as O)


[('44', 15),
 ('56', 13),
 ('49', 13),
 ('6222021234567890123', 12),
 ('25', 11),
 ('31', 11),
 ('27', 11),
 ('19', 11),
 ('54', 11),
 ('51', 10),
 ('26', 10),
 ('13857123456', 10),
 ('28', 10),
 ('76', 9),
 ('58', 9)]

In [55]:
fns_df = ModelError.get_fns_dataframe(results.model_errors, entity=["PERSON"])
fns_df[["full_text", "token", "annotation", "prediction"]].head(20)

No errors of type ErrorType.FN and entity ['PERSON'] were found


TypeError: 'NoneType' object is not subscriptable